# 群ごとの epoch 推移 — GroupDRO の `q`、hidden cohort、属性群

iterative 4 run（変調範囲 `fc` / `stage4+fc` × GroupDRO step size 1e-3 / 1e-2）で、stage01 / stage02 の
GroupDRO が**どの cohort に重みを寄せ、その cohort と属性群の val 性能がどう動いたか**を epoch ごとに見る。

問いは 4 つある。

1. `q` は step size ごとにどこまで偏るか。
2. hidden cohort の worst（最小 AUROC・最小 bACC・最大 loss）と gap（AUROC・bACC）は、stage の中で改善するか。
3. `q` の高い cohort と低い cohort は、stage の中で val の性能がどう違う動きをするか。
4. 属性群（sex・age group・race）の worst / gap と Eopp0 / Eopp1 / Eodds は、baseline と比べてどう動くか。

全 run が seed 42 の 1 本なので、ここで読めるのは推移の形であり、条件間の差の有意性ではない。
global 指標の推移は [`global_training_curves.ipynb`](global_training_curves.ipynb)、cohort の属性の構成と class weight は
[`cohort_composition.ipynb`](cohort_composition.ipynb) で扱う。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import STYLE_SHEET, run_dir, split_csv  # noqa: E402
from analysis.common.run_artifacts import read_wandb_history  # noqa: E402
from analysis.iterative_probe._shared import collect_epoch_rows, rank_cohorts_by_q  # noqa: E402

matplotlib.style.use(STYLE_SHEET)
pd.set_option("display.max_rows", 120)

## 対象 run と定数

run は条件順（fc 1e-3 → fc 1e-2 → stage4+fc 1e-3 → stage4+fc 1e-2）に並べる。baseline は
[global_training_curves](global_training_curves.ipynb) と同じ ERM run で、属性群の指標だけに参照線として重ねる
（hidden cohort は iterative の run ごとに作られるので、baseline には無い）。

cohort は 10 個（`NN` = 00〜09）で、**stage ごとに引き直される**。stage01 の cohort 03 と stage02 の cohort 03 は
別の集団なので、stage をまたいで cohort の線をつながない。

一方、同じ変調範囲の 2 run は warmup が一致するので、**stage01 の cohort は step size 1e-3 と 1e-2 で同じ集団**になる
（val の support が 10 cohort とも一致する）。stage01 は、同じ cohort に対して step size だけを変えた比較として読める。
stage02 の cohort は stage01 の終わりの model から作るので、step size ごとに違う。

In [ ]:
STUDY = "iterative_probe"
ITERATIVE_RUN_IDS = [
    "20260924T112035Z-spatial-lora-iterative-chexpert-fc-s42-b81a",
    "20260924T112045Z-spatial-lora-iterative-chexpert-fc-s42-4602",
    "20260924T112034Z-spatial-lora-iterative-chexpert-stage4-fc-s42-1caa",
    "20260924T112043Z-spatial-lora-iterative-chexpert-stage4-fc-s42-c755",
]
BASELINE_RUN_ID = "20260921T103036Z-resnet-chexpert-s42-5538"
BASELINE_CONDITION = "ResNet ERM"

COHORTS = range(10)
COHORT_STAGES = ["stage01", "stage02"]
MODULATIONS = ["fc", "stage4+fc"]
STEP_SIZES = [0.001, 0.01]
STEP_COLOR = {0.001: "#0173B2", 0.01: "#DE8F05"}
MODULATION_LINESTYLE = {"fc": "-", "stage4+fc": "--"}
BASELINE_STYLE = {"color": "#000000", "marker": "s", "linestyle": ":", "markersize": 3.5}

ATTRIBUTES = {"sex": "sex", "age_group_65": "age group (65)", "race": "race (all 6 codes)"}
GROUP_PERFORMANCE = {
    "worst_group_auroc": "worst-group AUROC",
    "auroc_gap": "AUROC gap",
    "worst_group_bacc": "worst-group bACC",
    "bacc_gap": "bACC gap",
}
EQUALIZED = {"Eopp1": "Eopp1 (TPR gap)", "Eopp0": "Eopp0 (TNR gap)", "Eodds": "Eodds"}

RESULTS = ROOT / "analysis" / STUDY / "results"
FIGURES = ROOT / "analysis" / STUDY / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)

## 入力の読み込み

iterative は `stages/<stage>/metrics/metrics.csv` を通し epoch 番号（`run_epoch`、warmup 0–1、stage01 2–6、stage02 7–11）
付きで読む（`_shared.collect_epoch_rows`）。cohort の worst / gap は、logger が記録した cohort ごとの生の値
（`val/hidden_{auroc,bacc,loss}_NN`）から `_shared.add_hidden_summaries` が計算した列を使う。

`train/group_dro/q_NN` は各 train epoch の終わりの `q` である。stage の頭で一様（0.1）に初期化される。

baseline は CSVLogger を付けずに回したため、W&B の transaction log から読む。

In [ ]:
iterative = pd.DataFrame([row for run_id in ITERATIVE_RUN_IDS for row in collect_epoch_rows(run_dir(STUDY, run_id))])
baseline_history = pd.DataFrame(read_wandb_history(run_dir(STUDY, BASELINE_RUN_ID)))
BUDGET = int(iterative["run_epoch"].max())
baseline = baseline_history.assign(
    run_id=BASELINE_RUN_ID, condition=BASELINE_CONDITION, stage="erm", run_epoch=baseline_history["epoch"].astype(int)
)
baseline_in_budget = baseline[baseline["run_epoch"] <= BUDGET]

first_run = iterative[iterative["run_id"] == ITERATIVE_RUN_IDS[0]]
stage_start = first_run.groupby("stage", sort=False)["run_epoch"].min().to_dict()
boundaries = [epoch - 0.5 for epoch in stage_start.values()][1:]
cohort_rows = iterative[iterative["stage"].isin(COHORT_STAGES)]
stage_start

cohort ごとの値を、run × stage × epoch × cohort の縦長の表にする。以降の cohort の図と表はこの表から作る。

In [ ]:
cohort_long = pd.concat(
    [
        cohort_rows[["run_id", "condition", "modulation", "step_size", "stage", "epoch", "run_epoch"]].assign(
            cohort=cohort,
            q=cohort_rows[f"train/group_dro/q_{cohort:02d}"],
            val_auroc=cohort_rows[f"val/hidden_auroc_{cohort:02d}"],
            val_bacc=cohort_rows[f"val/hidden_bacc_{cohort:02d}"],
            val_loss=cohort_rows[f"val/hidden_loss_{cohort:02d}"],
            val_support=cohort_rows[f"val/hidden_support_{cohort:02d}"],
        )
        for cohort in COHORTS
    ],
    ignore_index=True,
)
cohort_long.to_csv(RESULTS / "cohort_epoch_metrics.csv", index=False)
cohort_long.groupby(["condition", "stage"], sort=False)["val_support"].agg(["min", "max"])

## 1. `q` の偏り方

変調範囲を行、step size を列に分け、cohort ごとの `q` を描く。色は cohort 番号を表すが、stage01 と stage02 の
同じ色は別の cohort である。灰色の横線は一様な重み（0.1）。

In [ ]:
cohort_colors = plt.get_cmap("tab10").colors

figure, axes = plt.subplots(len(MODULATIONS), len(STEP_SIZES), figsize=(12, 7), sharex=True, sharey=True)
for row, modulation in enumerate(MODULATIONS):
    for column, step_size in enumerate(STEP_SIZES):
        axis = axes[row, column]
        part = cohort_long[(cohort_long["modulation"] == modulation) & (cohort_long["step_size"] == step_size)]
        for (stage, cohort), line in part.groupby(["stage", "cohort"], sort=False):
            axis.plot(
                line["run_epoch"],
                line["q"],
                marker="o",
                markersize=3,
                linewidth=1.4,
                color=cohort_colors[cohort],
                label=f"cohort {cohort:02d}" if stage == "stage01" else None,
            )
        axis.axhline(1 / len(COHORTS), color="#9a9a96", linewidth=0.8)
        for boundary in boundaries[1:]:
            axis.axvline(boundary, color="#9a9a96", linestyle="--", linewidth=0.8)
        axis.set_title(f"{modulation} / step size {step_size:g}")
        if column == 0:
            axis.set_ylabel("GroupDRO q (end of train epoch)")
        if row == len(MODULATIONS) - 1:
            axis.set_xlabel("epoch (stage01 → stage02)")
        axis.set_xticks(range(stage_start["stage01"], BUDGET + 1))

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.06))
figure.savefig(FIGURES / "cohort_q_curves.png", dpi=180, bbox_inches="tight")
plt.show()

stage の終わりの偏りを数値で見る。`max_q` は最大の重み、`weight_entropy` は `q` の entropy（一様なら log 10 = 2.303）、
`top3_q` は上位 3 cohort の重みの合計（一様なら 0.3）。

In [ ]:
stage_end = cohort_long.groupby(["condition", "stage"], sort=False)["epoch"].transform("max") == cohort_long["epoch"]
q_end = cohort_long[stage_end]
q_summary = q_end.groupby(["condition", "stage"], sort=False).agg(
    max_q=("q", "max"), top3_q=("q", lambda values: values.nlargest(3).sum()), min_q=("q", "min")
)
q_summary["weight_entropy"] = (
    iterative[iterative["stage"].isin(COHORT_STAGES)]
    .groupby(["condition", "stage"], sort=False)["train/group_dro/weight_entropy"]
    .last()
)
q_summary.round(3)

## 2. hidden cohort の worst と gap

cohort の worst（最小 AUROC・最小 bACC・最大 loss）と AUROC・bACC の gap（max − min）を、global の図と同じ配置（変調範囲を行、
色を step size）で描く。worst を取る cohort は epoch ごとに入れ替わりうる。baseline には cohort が無いので線は無い。
比較のため、同じ run の global val AUROC を細い灰色線で最小 AUROC の軸に重ねる。

In [ ]:
HIDDEN = {
    "val/hidden_min_auroc": "hidden min AUROC",
    "val/hidden_auroc_gap": "hidden AUROC gap",
    "val/hidden_min_bacc": "hidden min bACC",
    "val/hidden_bacc_gap": "hidden bACC gap",
    "val/hidden_max_loss": "hidden max loss",
}

figure, axes = plt.subplots(len(MODULATIONS), len(HIDDEN), figsize=(20, 7), sharex=True, sharey="col")
for row, modulation in enumerate(MODULATIONS):
    for column, (metric, title) in enumerate(HIDDEN.items()):
        axis = axes[row, column]
        part = iterative[iterative["modulation"] == modulation]
        for _, line in part.groupby("run_id", sort=False):
            step_size = line["step_size"].iloc[0]
            for stage in COHORT_STAGES:
                segment = line[line["stage"] == stage]
                axis.plot(
                    segment["run_epoch"],
                    segment[metric],
                    marker="o",
                    markersize=3.5,
                    linewidth=1.6,
                    color=STEP_COLOR[step_size],
                    label=f"step size {step_size:g}" if stage == "stage01" else None,
                )
                if metric == "val/hidden_min_auroc":
                    axis.plot(
                        segment["run_epoch"],
                        segment["val/auroc"],
                        linewidth=0.8,
                        color=STEP_COLOR[step_size],
                        alpha=0.35,
                        label=f"global AUROC ({step_size:g})" if stage == "stage01" else None,
                    )
        for boundary in boundaries[1:]:
            axis.axvline(boundary, color="#9a9a96", linestyle="--", linewidth=0.8)
        if row == 0:
            axis.set_title(title)
        if column == 0:
            axis.set_ylabel(modulation)
        if row == len(MODULATIONS) - 1:
            axis.set_xlabel("epoch (stage01 → stage02)")
        axis.set_xticks(range(stage_start["stage01"], BUDGET + 1))

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.05))
figure.savefig(FIGURES / "hidden_cohort_curves.png", dpi=180, bbox_inches="tight")
plt.show()

stage の頭（stage 内 epoch 0）と終わり（epoch 4）の差で、stage の中で worst が動いたかを見る。

In [ ]:
hidden_change = (
    iterative[iterative["stage"].isin(COHORT_STAGES)]
    .groupby(["condition", "stage"], sort=False)[list(HIDDEN)]
    .agg(lambda values: values.iloc[-1] - values.iloc[0])
    .rename(columns=HIDDEN)
)
hidden_change.round(3)

## 3. `q` の高い cohort と低い cohort の推移

`q` は iter ごとに動くので、1 時点の `q` と性能の相関では「重みを受けた量」と「その後の変化」を対応させにくい。
ここでは stage ごとに **`q` の高い 2 cohort と低い 2 cohort** を選び、その 4 cohort の stage 内の推移を並べて見る。

- 順位は **stage 内の epoch 終わりの `q` の平均**で付ける。stage を通して model がその cohort に掛けた重みの目安であり、
  stage 終わりの 1 時点より、学習に効いた量に近い。stage 終わりの `q` で選んだ場合との違いは下の表で確かめる。
- 性能は cohort ごとの val AUROC・bACC・loss。global の変動（全 cohort が同時に上下する分）と区別するため、
  10 cohort の平均を灰色の線で重ねる。選んだ cohort の線がこの平均から離れていくかを見る。
- step size 1e-3 は `q` がほぼ一様（0.06〜0.14）なので、「高い」「低い」の差は小さい。1e-2 との対照として並べる。

In [ ]:
PICK = 2
q_rank = rank_cohorts_by_q(iterative, PICK, len(COHORTS))
picked = q_rank[q_rank["pick"] != "middle"].sort_values(["condition", "stage", "rank_mean"], key=None)
picked.to_csv(RESULTS / "cohort_q_picked.csv", index=False)
picked.round(3)

平均で選んだ 4 cohort が、stage 終わりの `q` で選んでも同じ組になるかを数える（`same_pick` は 4 個中の一致数）。

In [ ]:
def picked_set(part: pd.DataFrame, rank: str) -> set[tuple[str, int]]:
    high = part.loc[part[rank] <= PICK, "cohort"]
    low = part.loc[part[rank] > len(COHORTS) - PICK, "cohort"]
    return {("high", cohort) for cohort in high} | {("low", cohort) for cohort in low}


q_rank.groupby(["condition", "stage"], sort=False).apply(
    lambda part: len(picked_set(part, "rank_mean") & picked_set(part, "rank_end")), include_groups=False
).rename("same_pick")

行が条件、列が指標（`q`・val AUROC・val bACC・val loss）。高い 2 cohort を赤系の実線、低い 2 cohort を青系の破線で描き、
凡例は `stage:cohort`。stage01 と stage02 で選ばれる cohort は別の集団なので、線は stage ごとに切る。
灰色の線は、その stage の 10 cohort の平均である。

In [ ]:
PICK_STYLE = {
    ("high", 1): {"color": "#C0392B", "linestyle": "-", "marker": "o"},
    ("high", 2): {"color": "#E67E22", "linestyle": "-", "marker": "s"},
    ("low", 1): {"color": "#1F618D", "linestyle": "--", "marker": "o"},
    ("low", 2): {"color": "#48C9B0", "linestyle": "--", "marker": "s"},
}
PICK_METRICS = {
    "q": "GroupDRO q",
    "val_auroc": "cohort val AUROC",
    "val_bacc": "cohort val bACC",
    "val_loss": "cohort val loss",
}
CONDITIONS = list(dict.fromkeys(cohort_long["condition"]))

picked_long = cohort_long.merge(
    picked[["condition", "stage", "cohort", "pick", "rank_mean"]], on=["condition", "stage", "cohort"]
)
# 高い群は順位 1, 2、低い群は下から 1, 2 に番号を振って線の見た目を決める。
picked_long["order"] = picked_long.groupby(["condition", "stage", "pick"])["rank_mean"].rank(method="dense").astype(int)
picked_long.loc[picked_long["pick"] == "low", "order"] = 3 - picked_long.loc[picked_long["pick"] == "low", "order"]
cohort_mean = (
    cohort_long.groupby(["condition", "stage", "run_epoch"], sort=False)[list(PICK_METRICS)].mean().reset_index()
)

figure, axes = plt.subplots(len(CONDITIONS), len(PICK_METRICS), figsize=(17, 13), sharex=True, sharey="col")
for row, condition in enumerate(CONDITIONS):
    for column, (metric, title) in enumerate(PICK_METRICS.items()):
        axis = axes[row, column]
        for stage in COHORT_STAGES:
            mean_line = cohort_mean[(cohort_mean["condition"] == condition) & (cohort_mean["stage"] == stage)]
            axis.plot(mean_line["run_epoch"], mean_line[metric], color="#9a9a96", linewidth=2.2, alpha=0.7)
            part = picked_long[(picked_long["condition"] == condition) & (picked_long["stage"] == stage)]
            for (pick, order, cohort), line in part.groupby(["pick", "order", "cohort"]):
                axis.plot(
                    line["run_epoch"],
                    line[metric],
                    markersize=3.5,
                    linewidth=1.5,
                    label=f"{pick} {stage[-2:]}:{cohort:02d}",
                    **PICK_STYLE[(pick, order)],
                )
        axis.axvline(boundaries[1], color="#9a9a96", linestyle="--", linewidth=0.8)
        if metric == "q":
            axis.axhline(1 / len(COHORTS), color="#9a9a96", linewidth=0.8, linestyle=":")
        if row == 0:
            axis.set_title(title)
        if column == 0:
            axis.set_ylabel(condition)
        if row == len(CONDITIONS) - 1:
            axis.set_xlabel("epoch (stage01 → stage02)")
        axis.set_xticks(range(stage_start["stage01"], BUDGET + 1))
    axes[row, -1].legend(fontsize=7, ncol=2, loc="upper left", bbox_to_anchor=(1.02, 1.0))

figure.savefig(FIGURES / "cohort_q_picked_curves.png", dpi=180, bbox_inches="tight")
plt.show()

同じ推移を、stage 内の変化量（epoch 4 − epoch 0）で表にする。`vs_mean` は、その cohort の変化量から 10 cohort の平均の
変化量を引いたもので、global の上下を除いた「その cohort だけの動き」になる。

In [ ]:
def stage_change(frame: pd.DataFrame) -> pd.DataFrame:
    ordered = frame.sort_values("epoch")
    grouped = ordered.groupby(["condition", "stage", "cohort"], sort=False)
    return (
        grouped[["val_auroc", "val_bacc", "val_loss"]].last() - grouped[["val_auroc", "val_bacc", "val_loss"]].first()
    )


change = stage_change(cohort_long)
mean_change = change.groupby(["condition", "stage"], sort=False).transform("mean")
picked_change = (
    change.join((change - mean_change).add_suffix("_vs_mean"))
    .reset_index()
    .merge(picked[["condition", "stage", "cohort", "pick", "q_mean"]], on=["condition", "stage", "cohort"])
    .sort_values(["condition", "stage", "q_mean"], ascending=[True, True, False])
)
picked_change.round(3)

高い 2 cohort と低い 2 cohort で、`vs_mean` の平均を比べる。高い群のほうが AUROC の `vs_mean` が大きく、loss の `vs_mean` が
小さければ、重みを受けた cohort が global の動きを超えて良くなったことになる。

In [ ]:
picked_change.groupby(["condition", "stage", "pick"], sort=False)[
    ["val_auroc_vs_mean", "val_bacc_vs_mean", "val_loss_vs_mean"]
].mean().unstack("pick").round(3)

## 4. 属性群の公平性指標

学習中に毎 epoch 記録している属性群の指標（`val/<属性>/<指標>`）を baseline と並べる。値は logger が val 全体で
計算したもので、ここでは計算し直さない。

- worst / gap は属性内の群の AUROC・bACC の min と max − min。
- `Eopp1` は TPR、`Eopp0` は TNR の群間の max − min、`Eodds` は (TPR gap + FPR gap) / 2。陽性の判定は argmax（閾値 0.5）。
  二値分類では TNR = 1 − FPR なので FPR gap は `Eopp0` と同じ値になり、**`Eodds` = (`Eopp1` + `Eopp0`) / 2** である。
  図には 3 つとも描くが、`Eodds` に独立な情報は無い。

**race は 6 コードすべてを群にしている。** README の評価軸（White / Asian / Black に絞る）とは定義が違う。
学習側は群ごとの値を残していないので、epoch 推移では絞り直せない。小さな群が worst と gap を決めうるので、
val の群の大きさを先に確かめる。

In [ ]:
val_frame = pd.read_csv(split_csv("chexpert", "val"))
val_race = val_frame[~val_frame["race_missing"].astype(bool)]
val_race.groupby("race")["target"].agg(n="size", positives="sum")

属性群の値を、run × epoch × 属性 × 指標の縦長の表にする。baseline は 12 epoch の予算に切る。

In [ ]:
def attribute_long(frame: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for attribute in ATTRIBUTES:
        for metric in [*GROUP_PERFORMANCE, *EQUALIZED]:
            parts.append(
                frame[["run_id", "condition", "stage", "run_epoch"]].assign(
                    attribute=attribute, metric=metric, value=frame[f"val/{attribute}/{metric}"]
                )
            )
    return pd.concat(parts, ignore_index=True)


attribute_epoch = pd.concat([attribute_long(iterative), attribute_long(baseline_in_budget)], ignore_index=True)
attribute_epoch.to_csv(RESULTS / "attribute_fairness_epoch.csv", index=False)
len(attribute_epoch)

4 条件を同じ軸に重ねる。色は step size、線種は変調範囲（実線 `fc`、破線 `stage4+fc`）、黒の点線が baseline。
行が属性、列が指標で、縦軸は列の中で揃えないで属性ごとに取る（sex と race で値の桁が違う）。

In [ ]:
def plot_attribute_metrics(metrics: dict[str, str], path: Path) -> None:
    figure, axes = plt.subplots(len(ATTRIBUTES), len(metrics), figsize=(4 * len(metrics), 9.5), sharex=True)
    for row, (attribute, attribute_title) in enumerate(ATTRIBUTES.items()):
        for column, (metric, metric_title) in enumerate(metrics.items()):
            axis = axes[row, column]
            key = f"val/{attribute}/{metric}"
            axis.plot(
                baseline_in_budget["run_epoch"], baseline_in_budget[key], label=BASELINE_CONDITION, **BASELINE_STYLE
            )
            for _, line in iterative.groupby("run_id", sort=False):
                modulation, step_size = line["modulation"].iloc[0], line["step_size"].iloc[0]
                axis.plot(
                    line["run_epoch"],
                    line[key],
                    marker="o",
                    markersize=3,
                    linewidth=1.4,
                    color=STEP_COLOR[step_size],
                    linestyle=MODULATION_LINESTYLE[modulation],
                    label=line["condition"].iloc[0],
                )
            for boundary in boundaries:
                axis.axvline(boundary, color="#9a9a96", linestyle="--", linewidth=0.8)
            if row == 0:
                axis.set_title(metric_title)
            if column == 0:
                axis.set_ylabel(attribute_title)
            if row == len(ATTRIBUTES) - 1:
                axis.set_xlabel("epoch (warmup → stage01 → stage02)")
            axis.set_xticks(range(BUDGET + 1))
    handles, labels = axes[0, 0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.03))
    figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()

### worst group と gap

In [ ]:
plot_attribute_metrics(GROUP_PERFORMANCE, FIGURES / "attribute_group_performance_curves.png")

### Eopp0 / Eopp1 / Eodds

In [ ]:
plot_attribute_metrics(EQUALIZED, FIGURES / "attribute_equalized_curves.png")

### 区間ごとの平均

1 epoch ごとの振れが大きいので、区間の平均で baseline と比べる。warmup（epoch 0–1）、stage01（2–6）、stage02（7–11）で
平均を取り、baseline も同じ epoch で区切る。値の小さいほうが良い指標（gap・Eopp・Eodds）と、大きいほうが良い
指標（worst）が混ざっている点に注意する。

In [ ]:
PHASES = ["warmup", "stage01", "stage02"]
phase_of_epoch = first_run.set_index("run_epoch")["stage"]
attribute_epoch["phase"] = attribute_epoch["run_epoch"].map(phase_of_epoch)
phase_mean = (
    attribute_epoch.groupby(["attribute", "metric", "condition", "phase"], sort=False)["value"]
    .mean()
    .unstack("phase")[PHASES]
)
phase_mean.to_csv(RESULTS / "attribute_fairness_phase_mean.csv")
phase_mean.loc[["sex", "age_group_65"]].round(3)

In [ ]:
phase_mean.loc[["race"]].round(3)

## まとめ

数値の正本は `results/cohort_epoch_metrics.csv`、`results/attribute_fairness_epoch.csv`、
`results/attribute_fairness_phase_mean.csv`。読み取りの記録は
[reports/subgroup_training_curves.md](reports/subgroup_training_curves.md) に残す。

**観察**

- **step size 1e-3 では `q` はほぼ一様のまま。** stage 終わりの max `q` は 0.12〜0.14、entropy は 2.27〜2.29（一様で 2.303）。
  1e-2 では上位 3 cohort が 0.72〜0.82 を持ち、stage 終わりでもまだ偏り続けている。
- **`q` の高い cohort は、stage を通して val でも弱い cohort である。** stage 内平均の `q` で選んだ上位 2 cohort は、
  16 個（4 run × 2 stage × 2 cohort）中 15 個で、val AUROC が 10 cohort の平均より低い位置を最初から最後まで保つ。
  残る 1 個（fc / 1e-3 の 02:05）も平均付近に留まる。下位 2 cohort は多くが平均より高いが、平均より低いまま推移する
  cohort もある（fc / 1e-3 の 01:07、stage4+fc の 01:02）。選び方を stage 終わりの `q` に変えても、4 cohort 中 3〜4 個は同じになる。
- **重みを受けても、弱い cohort は追いつかない。** 10 cohort の平均の変化を引いた stage 内の AUROC の変化は、
  上位 2 cohort の平均で −0.014〜+0.008、下位 2 cohort で −0.007〜+0.010 で、上位が下位を上回るのは 8 通り中 4 通りに留まる。
  step size 1e-2 で `q` が 0.2〜0.5 まで上がった cohort でも同じである。
- **loss だけは、`q` の高い cohort で平均より下がる。** 平均との差は 8 通り中 5 通りで −0.09〜−0.17（下位は −0.03〜+0.07）。
  ただし step size 1e-3（上位の `q` でも stage 終わりで 0.12〜0.14）でも同じ大きさで起きるので、重みの効果とは読めない。
  `q` の順位を決める train の損失が高い cohort ほど、val loss も戻りやすいと読むほうが自然である。
- **hidden cohort の worst は stage 内でほぼ動かない。** 最小 AUROC は 0.75〜0.79 に留まり、stage の頭と終わりの差は
  −0.026〜+0.025。上がり幅が大きいのは stage01 の 1e-3 で、`q` の偏りの効果とは読めない。
  bACC gap は stage4+fc の stage01 で 0.027 / 0.029 縮むが、step size によらない。
- **age group の Eopp0 / Eopp1 / Eodds は stage に入ると大きく下がる。** stage01 / stage02 の平均で Eodds は 0.085〜0.112、
  baseline は 0.16〜0.17。step size 1e-3（`q` がほぼ一様）でも同じだけ下がる。
  同じ区間の age group の AUROC gap は 0.032〜0.050 で baseline（0.040〜0.046）と変わらない。
- **sex の Eopp は stage02 で baseline よりやや大きい。** stage02 の平均の Eodds は 1e-2 が 0.019 / 0.020、1e-3 が 0.013 / 0.014
  （fc / stage4+fc）、baseline は 0.009。step size の差は epoch 間の振れより小さい。
- **worst-group AUROC は sex・age group とも baseline より 0.01〜0.025 低い。** gap はほぼ変わらず、global AUROC の低下
  （global_training_curves）が群の全体に出ている形である。
- **race（6 コード）は epoch 間の振れが条件差より大きい。** val で code 5 は 30 件（陽性 3）しかない。

**解釈への引き継ぎ**

- age group の Eopp が `q` の偏りと無関係に下がる理由は、cohort の構成と cohort × class の class weight を見る
  [`cohort_composition.ipynb`](cohort_composition.ipynb) で扱う。